# **Anomaly Detection**

Prerequisites: KDE (density view), Decision Trees (Isolation Forest),
Bias-Variance (contamination parameter as a tuning knob).

## 1. Statistical methods - Z-score and IQR, derived

**Z-score**: $z_i = \dfrac{x_i-\bar x}{s}$; flag $|z_i|>3$ (under a Gaussian
assumption, $P(|Z|>3)\approx0.0027$ - a principled threshold, not
arbitrary, directly from the standard normal CDF).
**IQR method**: flag $x_i < Q_1-1.5\,\text{IQR}$ or $x_i>Q_3+1.5\,\text{IQR}$,
$\text{IQR}=Q_3-Q_1$ - robust to non-Gaussian data since it uses quantiles,
not mean/std (which are themselves outlier-sensitive).

### Worked numerical example

Data: $\{12,13,14,15,16,17,18,45\}$. $\bar x = 18.75$, population
$s\approx10.10$.

$z_{45} = (45-18.75)/10.10 \approx 2.60$ - below the 3.0 threshold despite
being visibly an outlier (mean/std are themselves distorted by the
outlier - the exact motivation for preferring IQR here). 

IQR: sorted data
gives $Q_1=13.75$, $Q_3=17.25$, $\text{IQR}=3.5$; upper fence
$=17.25+1.5(3.5)=22.5$ → $45>22.5$ → **flagged** by IQR, correctly, while
the Z-score method misses it at the standard threshold.

## 2. Isolation Forest - derivation of the core idea

Build random trees that recursively split on random features at random
thresholds. Anomalies, being "few and different," require **fewer splits**
to isolate into their own leaf than normal points do. Anomaly score based
on average path length $h(x)$ across trees:
$$s(x) = 2^{-h(x)/c(n)}, \qquad c(n)=2H(n-1)-\frac{2(n-1)}{n}\ \text{(average path length of unsuccessful BST search, harmonic-number-based normalizer)}$$
$s(x)$ close to 1 → likely anomaly (short average path); close to 0.5 →
likely normal.

## 3. One-Class SVM - brief

Finds a boundary (in kernel-feature space) enclosing "normal" data as
tightly as possible while allowing a specified fraction ($\nu$) of points
to fall outside - directly reuses the kernel-trick machinery already
derived in `03_Supervised_Learning\Classification\15_kernel_methods.ipynb`.

## 4. Python - verify the Z-score/IQR worked example, run Isolation Forest

In [ ]:
import numpy as np
x = np.array([12,13,14,15,16,17,18,45])
z = (x - x.mean())/x.std()
print(z)  # z_45 ~2.42 -- below 3.0, matches hand calc

q1, q3 = np.percentile(x, [25,75])
iqr = q3-q1
upper_fence = q3 + 1.5*iqr
print(q1, q3, iqr, upper_fence, x[x>upper_fence])  # correctly flags 45

from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.1, random_state=0).fit(x.reshape(-1,1))
print(iso.predict(x.reshape(-1,1)))  # -1 for the outlier, 1 for normal points

[-0.66855445 -0.56950935 -0.47046424 -0.37141914 -0.27237404 -0.17332893
 -0.07428383  2.59993397]
13.75 17.25 3.5 22.5 [45]
[ 1  1  1  1  1  1  1 -1]
